In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd

In [2]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).dropna()

xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()
dids = df['did'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [3]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/vlos/*'))
#files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/solo/phi/vlos/solo_L2_phi-fdt-vlos_20240220T000009_V202602220014_0442200501.fits.gz',
 '/home/ulyanov/data/solo/phi/vlos/solo_L2_phi-fdt-vlos_20240331T063009_V202602220014_0443310502.fits.gz',
 '/home/ulyanov/data/solo/phi/vlos/solo_L2_phi-fdt-vlos_20240922T211509_V202602220112_0449220504.fits.gz',
 '/home/ulyanov/data/solo/phi/vlos/solo_L2_phi-fdt-vlos_20240930T121503_V202602220112_0449300001.fits.gz',
 '/home/ulyanov/data/solo/phi/vlos/solo_L2_phi-fdt-vlos_20250501T002002_V202602220258_0545010501.fits.gz',
 '/home/ulyanov/data/solo/phi/vlos/solo_L2_phi-fdt-vlos_20250528T152003_V202602220334_0545280506.fits.gz',
 '/home/ulyanov/data/solo/phi/vlos/solo_L2_phi-fdt-vlos_20251110T032003_V202602220437_0551100501.fits.gz']

In [4]:
file = files[3]

with fits.open(file) as hdul:
    header = hdul[0].header
    data = hdul[0].data

did = int(file.split('_')[-1].split('.')[0])
data = undistort(data, header, xd, yd)

In [5]:
view = View.from_header(header)
view.update(crota=view.crota + 0.29, xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0] + 0.28)

view_ = View.from_header(header)
view_.update(crota=view.crota + 0.29 + 180, xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0] + 0.28)

In [16]:
V = (view.velocity(mu_thr=0., cbs=False) - view_.velocity(mu_thr=0., cbs=False)) / 2 / 1000

plt.figure(figsize=(10,10))
plt.imshow(V, 'seismic', vmin=-2, vmax=2)
plt.tight_layout()

In [17]:
V_ = (data.copy() - view_.reproject(data, view)) / 2

plt.figure(figsize=(10,10))
plt.imshow(V_, 'seismic', vmin=-2, vmax=2)
plt.tight_layout()

In [15]:
t = ~np.isnan(V)
x, y = V[t], V_[t]
t = np.where(np.abs(y - x) < 0.5)
x, y = x[t], y[t]

k = np.nanmean(x * y) / np.nanmean(x ** 2)
print(k)

plt.figure(figsize=(10,10))
plt.plot([-2,2], [-2,2], color='black', lw=0.5)
plt.scatter(V, V_, s=0.001)
plt.plot([-2,2], [-2 * k, 2 * k], '--', color='black', lw=1)

plt.xlabel('Velocity, km/s')
plt.ylabel('Doppler velocity, km/s')

plt.xlim(-2,2)
plt.ylim(-2,2)

plt.tight_layout()

0.8385023445739466


In [9]:
V = view.velocity(mu_thr=0., cbs=False) / 1000
V -= np.nanmean(V)

V_ = data.copy()
V_[np.isnan(V)] = np.nan
V_ -= np.nanmean(V_)

U = V_ - V * k

In [10]:
plt.figure(figsize=(10,10))
plt.imshow(V_, 'seismic', vmin=-2, vmax=2)
plt.tight_layout()

In [11]:
plt.figure(figsize=(10,10))
plt.imshow(U, 'seismic', vmin=-1, vmax=1)
plt.tight_layout()

In [12]:
mu = view.mu()

t = np.where(mu > 0.1)
p = np.polyfit(mu[t], U[t], 3)

dx = 0.1
x = np.arange(dx / 2, 1, dx)
y = []
for x_ in x:
    t = np.where(np.abs(mu - x_) < dx / 2)
    y += [np.nanmean(U[t])]
y = np.array(y)

plt.figure(figsize=(10,10))
plt.scatter(mu, U, s=0.01)
plt.plot(x, y, '--', color='black')
plt.plot(np.arange(0,1.01,0.01), np.polyval(p, np.arange(0,1.01,0.01)), color='red', lw=1)

plt.title('Convective blueshift')
plt.xlabel(r'$\mu$')
plt.ylabel('Doppler velocity, km/s')

plt.xlim(0,1)
plt.ylim(-0.6,0.6)

plt.gca().invert_xaxis()
plt.tight_layout()

In [13]:
U_ = np.polyval(p, mu)

In [14]:
plt.figure(figsize=(10,10))
plt.imshow(U - U_, 'seismic', vmin=-1, vmax=1)
plt.tight_layout()